# 01 — Data Exploration & Server Capacity Profiling

## Portfolio objective

Establish the structure, quality and analytical boundaries of the Alibaba GPU Cluster Trace 2026 server-hourly dataset before forecasting or optimization.

### Questions answered

- What data is available?
- What are the server-level fields?
- How much GPU/CPU capacity is observed?
- How many servers and clusters are represented?
- Does the dataset contain direct workload utilization?

> **Important:** this dataset provides server capacity attributes. It does not provide direct GPU utilization, requested GPU demand, pod allocation or job-level workload measures.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cpoe.config import SERVER_DATASET
from cpoe.data.loader import discover_day_partitions
import pandas as pd

print("Server dataset:", SERVER_DATASET)
print("Exists:", SERVER_DATASET.exists())

days = discover_day_partitions(SERVER_DATASET)
print("Days discovered:", len(days))
print("First partitions:", [p.name for p in days[:5]])


In [ ]:
first_day = days[0]
hours = sorted(
    first_day.glob("hour=*"),
    key=lambda p: int(p.name.split("=")[1])
)
first_hour = hours[0]
parquet = sorted(first_hour.glob("*.parquet"))[0]

sample = pd.read_parquet(parquet)
print("Sample shape:", sample.shape)
print("Columns:", sample.columns.tolist())
display(sample.head())
display(sample.dtypes)


In [ ]:
from cpoe.data.loader import REQUIRED_COLUMNS, validate_schema

validate_schema(sample)
print("Required analytical fields validated:", REQUIRED_COLUMNS)

for col in ["gpu_count", "cpu_capacity_cores"]:
    sample[col] = pd.to_numeric(sample[col], errors="coerce")

display(sample[["gpu_count", "cpu_capacity_cores"]].describe())


## Analytical conclusion

The server-hourly data supports a defensible **capacity-footprint analysis**.

It does not support a claim that we are forecasting actual workload utilization. That distinction is carried through the remaining notebooks.
